In [1]:
# Import libraries and modules that are necessary for the script
import pandas as pd
from sodapy import Socrata
from sqlalchemy import create_engine
from sqlalchemy.types import Integer, BigInteger, Float, String, Text
from sqlalchemy.engine import URL
import os
from dotenv import load_dotenv

### Pull data from [PLUTO](https://data.cityofnewyork.us/City-Government/Primary-Land-Use-Tax-Lot-Output-PLUTO-/64uk-42ks/about_data) with Socrata API

In [2]:

# Define the data types for each column
column_dtypes = {
    # String columns
    'borough': 'string',
    'block': 'string',
    'lot': 'string',
    'cd': 'string',  # Community District
    'ct2010': 'string',  # Census Tract 2010
    'cb2010': 'string',  # Census Block 2010
    'schooldist': 'string',
    'council': 'string',
    'zipcode': 'string',
    'firecomp': 'string',
    'policeprct': 'string',
    'healtharea': 'string',
    'sanitboro': 'string',
    'sanitsub': 'string',
    'address': 'string',
    'zonedist1': 'string',
    'splitzone': 'string',
    'bldgclass': 'string',
    'landuse': 'string',
    'easements': 'string',
    'ownername': 'string',
    'areasource': 'string',
    'ext': 'string',
    'proxcode': 'string',
    'irrlotcode': 'string',
    'lottype': 'string',
    'bsmtcode': 'string',
    'zonemap': 'string',
    'sanborn': 'string',
    'taxmap': 'string',
    'appdate': 'string',
    'dcpedited': 'string',
    'spdist1': 'string',
    'zonedist2': 'string',
    'zmcode': 'string',
    'edesignum': 'string',
    'histdist': 'string',
    'firm07_flag': 'string',
    'pfirm15_flag': 'string',
    'zonedist3': 'string',
    'spdist2': 'string',
    'overlay2': 'string',
    'landmark': 'string',
    'condono': 'string',
    'version': 'string',
    
    # Integer columns (non-nullable)
    'numbldgs': 'int64',
    'yearbuilt': 'int64',
    'yearalter1': 'int64',
    'yearalter2': 'int64',
    'borocode': 'int64',
    'plutomapid': 'int64',
    
    
    # Float columns (non-nullable)
    'lotarea': 'float64',
    'bldgarea': 'float64',
    'comarea': 'float64',
    'resarea': 'float64',
    'officearea': 'float64',
    'retailarea': 'float64',
    'garagearea': 'float64',
    'strgearea': 'float64',
    'factryarea': 'float64',
    'otherarea': 'float64',
    'assessland': 'float64',
    'assesstot': 'float64',
    'exempttot': 'float64',
    'numfloors': 'float64',
    'builtfar': 'float64',
    'residfar': 'float64',
    'commfar': 'float64',
    'facilfar': 'float64',
    'ltdheight': 'string'
}

# Function to convert data types with logging
def convert_data_types(df, dtypes):
    for col, dtype in dtypes.items():
        if col in df.columns:
            try:
                df[col] = df[col].astype(dtype)
            except ValueError as e:
                print(f"Error converting column '{col}' to {dtype}: {e}")
                # Optionally, handle the error (e.g., fill with NaN or drop rows)
    return df

# Initialize variables
df_list = []
limit = 10000
offset = 0

# Set your domain and app token (you'll need to create an app on Socrata)
domain = 'data.cityofnewyork.us'
app_token = os.getenv('APP_TOKEN')

# Create a client object
client = Socrata(domain, app_token)

# Define the dataset ID for PLUTO
dataset_id = "64uk-42ks"

# Function to fetch data from the PLUTO dataset
def fetch_pluto_data(client, dataset_id, params):
    try:
        results = client.get(dataset_id, **params)
        return results
    except Exception as e:
        print(f"Error fetching data: {e}")
        return []

while True:
    try:
        # Update params with the current offset
        params = {
            "$limit": limit,
            "$offset": offset,
            "$order": ":id"  # Ordering ensures consistent pagination
        }
        
        page_data = fetch_pluto_data(client, dataset_id, params=params)
        
        if not page_data:
            break
            
        df_page = pd.DataFrame(page_data)
        df_page = convert_data_types(df_page, column_dtypes)  # Convert data types after fetching each page
        df_list.append(df_page)
        
        # Increment offset by the limit to get the next page
        offset += limit
        print(f"Retrieved {offset} records...")
    except Exception as e:
        print(f"Error processing data at offset {offset}: {e}")
        break

df = pd.concat(df_list, ignore_index=True)
# print(df.head())

# Verify the data types
# print(df.dtypes)

# Check the number of rows
print(f"Total rows retrieved: {len(df)}")

Retrieved 10000 records...
Retrieved 20000 records...
Retrieved 30000 records...
Retrieved 40000 records...
Retrieved 50000 records...
Retrieved 60000 records...
Retrieved 70000 records...
Retrieved 80000 records...
Retrieved 90000 records...
Retrieved 100000 records...
Retrieved 110000 records...
Retrieved 120000 records...
Retrieved 130000 records...
Retrieved 140000 records...
Retrieved 150000 records...
Retrieved 160000 records...
Retrieved 170000 records...
Retrieved 180000 records...
Retrieved 190000 records...
Retrieved 200000 records...
Error converting column 'numbldgs' to int64: cannot convert float NaN to integer
Retrieved 210000 records...
Retrieved 220000 records...
Error converting column 'numbldgs' to int64: cannot convert float NaN to integer
Retrieved 230000 records...
Error converting column 'numbldgs' to int64: cannot convert float NaN to integer
Retrieved 240000 records...
Retrieved 250000 records...
Error converting column 'numbldgs' to int64: cannot convert float 

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 858602 entries, 0 to 858601
Data columns (total 97 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   borough               858602 non-null  string 
 1   block                 858602 non-null  string 
 2   lot                   858602 non-null  string 
 3   cd                    857211 non-null  string 
 4   ct2010                857174 non-null  string 
 5   cb2010                857174 non-null  string 
 6   schooldist            856480 non-null  string 
 7   council               857209 non-null  string 
 8   zipcode               856468 non-null  string 
 9   firecomp              856461 non-null  string 
 10  policeprct            856475 non-null  string 
 11  healtharea            856475 non-null  string 
 12  sanitboro             856278 non-null  string 
 13  sanitsub              856160 non-null  string 
 14  address               858134 non-null  string 
 15  zonedist1  

### Write data to database table

In [4]:
def get_sqlalchemy_type(pandas_type, column_name):
    """Maps pandas dtypes to SQLAlchemy types with PLUTO-specific overrides."""
    
    # 1. Manual Overrides for PLUTO Identifiers
    # These must be strings to preserve leading zeros or precision
    string_fields = ['borough', 'block', 'lot', 'cd', 'ct2010', 'cb2010', 'schooldist',
       'council', 'zipcode', 'firecomp', 'policeprct', 'healtharea',
       'sanitboro', 'sanitsub', 'address', 'zonedist1', 'splitzone',
       'bldgclass', 'landuse', 'ownername', 'areasource', 'ext', 'proxcode', 'irrlotcode', 'lottype',
       'bsmtcode', 'borocode', 'bbl', 'tract2010', 'zonemap', 'sanborn', 'taxmap', 'plutomapid',
       'version', 'sanitdistrict', 'healthcenterdistrict', 'bct2020',
       'bctcb2020', 'overlay1', 'ownertype', 'appbbl',
       'spdist1', 'zonedist2', 'zmcode', 'landmark', 'edesignum', 'histdist',
       'zonedist3', 'spdist2', 'overlay2',
       'ltdheight', 'condono', 'zonedist4']
    if any(field in column_name for field in string_fields):
        return String(255)
    
    # 2. Automated Mapping based on Pandas Dtypes
    if "int64" in str(pandas_type):
        return BigInteger
    elif "int" in str(pandas_type):
        return Integer
    elif "float" in str(pandas_type):
        return Float
    else:
        return Text  # Default for everything else (object/strings)

# 1. Setup connection (Replace with your actual password and db name)

# 1. Database Connection
url_object = URL.create(
    "postgresql",
    username=os.getenv('DB_USERNAME'),
    password=os.getenv('DB_PASSWORD'),  # SQLAlchemy handles the @ character for you here
    host=os.getenv('DB_HOST'),
    port=int(os.getenv('DB_PORT')),
    database=os.getenv('DB_NAME')
)

engine = create_engine(url_object)

# 2. Load the CSV
# We use low_memory=False because the PLUTO dataset has many columns
# df = pd.read_csv('EDA-PLUTO.csv', low_memory=False)

sql_dtypes = {col: get_sqlalchemy_type(dtype, col) for col, dtype in df.dtypes.items()}

# 3. Clean column names (PostgreSQL prefers lowercase, no spaces)
df.columns = [c.lower().replace(' ', '_') for c in df.columns]

# 4. Automate the table creation and input
# 'replace' will drop the table and recreate it if it already exists
df.to_sql('nyc_pluto_data', engine, if_exists='replace', index=False, dtype=sql_dtypes)


print("Table created and data uploaded successfully!")


Table created and data uploaded successfully!
